# 2_1-2 Analysis Notebook


## Table of Contents
1. Load/prepare
2. 2_1 intermediate tables
3. 2_2 metrics
4. Assemble summary
5. Export CSV/XLSX
6. Final preview/checks

## What to run
- Run all cells top-to-bottom after generating model outputs (`2_1-2_with_answers_*.csv`).
- Required inputs for this notebook:
  `2_1-2_with_answers_claude-opus_4_6.csv`,
  `2_1-2_with_answers_gemini.csv`,
  `2_1-2_with_answers_gpt-5_2.csv`.

## What not to edit (unless you intentionally change methodology)
- Matching thresholds and triage logic.
- Field names used in intermediate tables.
- Final metric formulas and aggregation rules.

## Documentation
- Methodology and field reference: [2_1-2_Rule_Classification_Eval.md](/C:/Users/ritaMZ/PycharmProjects/bravo_benchmark/eval/2_rule_understanding/2_1-2_rule_classific_decompos/2_1-2_Rule_Classification_Eval.md)
- Inputs/outputs guide: [2_1-2_inputs_outputs.md](/C:/Users/ritaMZ/PycharmProjects/bravo_benchmark/eval/2_rule_understanding/2_1-2_rule_classific_decompos/2_1-2_inputs_outputs.md)

---


# 1. Load/prepare

In [ ]:
import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Sentence-BERT import OK")

In [ ]:
!pip install pandas requests tqdm openpyxl

## 1.1 Load source answers (Claude)

### 1.2 Parse and normalize answer fields

In [ ]:
import pandas as pd
import re

# INPUT = "output_with_answers_recovered.csv"
# INPUT = "1_2_with_answers_claude-opus-4.6.csv"
INPUT = "2_1-2_with_answers_claude-opus_4_6.csv"

COL_TEMPLATE = "template_id"
COL_GT = "ground_truth_answer"
COL_PRED = "model_answer"

from pathlib import Path
if not Path(INPUT).exists():
    raise FileNotFoundError(f"Claude Opus input CSV not found: {INPUT}")
df = pd.read_csv(INPUT)


### Imports and config

In [ ]:
import numpy as np
import pandas as pd
import re
from pathlib import Path
from typing import Any

# Claude section config (rule-understanding tasks)
MODEL_NAME = "Claude Opus 4.6"
COL_LAYER = "layer_id"
COL_GT = "ground_truth_answer"
COL_PRED = "model_answer"

RULE_CLASSIFICATION_LAYER = "2_1"
RULE_UNDERSTANDING_LAYER = "2_2"

# Matching and triage thresholds.
RULE_WEAK_MATCH_THRESHOLD = 0.58
RULE_MATCH_THRESHOLD = 0.71
RULE_STRONG_MATCH_THRESHOLD = 0.82
RULE_AUTO_ACCEPT_THRESHOLD = RULE_STRONG_MATCH_THRESHOLD
RULE_AUTO_REJECT_THRESHOLD = RULE_WEAK_MATCH_THRESHOLD

# Review/export files for layer 2_1.
RULE_CLASSIFICATION_CANDIDATE_CSV = "2_1-2_rule_classification_candidate_matches.csv"
RULE_ALL_PAIRWISE_CSV = "2_1-2_all_pairwise_similarity.csv"
RULE_GT_COVERAGE_CSV = "2_1-2_gt_coverage_summary.csv"
RULE_SAMPLE_LEVEL_METRICS_CSV = "2_1-2_sample_level_metrics.csv"


In [ ]:
import math
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

RULE_UNDERSTANDING_TITLE = "Rule understanding (Rule classification)"
RULE_UNDERSTANDING_SUMMARY_CSV = "2_1-2_rule_understanding_summary_table.csv"
RULE_UNDERSTANDING_SUMMARY_XLSX = "2_1-2_rule_understanding_summary_table.xlsx"

RULE_MODEL_INPUTS = {
    "Claude Opus 4.6": "2_1-2_with_answers_claude-opus_4_6.csv",
    "Gemini 3 Pro Preview": "2_1-2_with_answers_gemini.csv",
    "GPT-5.2": "2_1-2_with_answers_gpt-5_2.csv",
}
RULE_MODEL_COLUMNS = ["Claude Opus 4.6", "Gemini 3 Pro Preview", "GPT-5.2"]

# Input schema in this notebook is standardized to:
# - ground_truth_answer
# - model_answer


### 1.3 Prepare base dataframe and normalize values

In [ ]:
def _ensure_rule_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_local = df_in.copy()

    required = [COL_LAYER, COL_GT, COL_PRED]
    missing = [c for c in required if c not in df_local.columns]
    if missing:
        raise KeyError(f"Missing required columns for rule routing/eval: {missing}")

    df_local[COL_LAYER] = df_local[COL_LAYER].fillna("").astype(str).str.strip()
    df_local[COL_GT] = df_local[COL_GT].fillna("").astype(str)
    df_local[COL_PRED] = df_local[COL_PRED].fillna("").astype(str)

    if "generated_question_id" not in df_local.columns:
        df_local["generated_question_id"] = pd.NA

    if "row_index" not in df_local.columns:
        df_local["row_index"] = df_local.index

    return df_local


df_eval = _ensure_rule_columns(df)

# -------- Routing (layer_id-based) --------
# layer_id == "2_1" -> Rule classification
# layer_id == "2_2" -> Rule understanding (image description)
df_eval["_layer_id_norm"] = df_eval[COL_LAYER].fillna("").astype(str).str.strip()

rule_classification_mask = df_eval["_layer_id_norm"] == RULE_CLASSIFICATION_LAYER
rule_understanding_mask = df_eval["_layer_id_norm"] == RULE_UNDERSTANDING_LAYER

df_rule_classification = df_eval[rule_classification_mask].copy()
df_rule_understanding = df_eval[rule_understanding_mask].copy()

print("Routing summary:")
print(f"- Rule classification ({RULE_CLASSIFICATION_LAYER}): {len(df_rule_classification)} rows")
print(f"- Rule understanding ({RULE_UNDERSTANDING_LAYER}): {len(df_rule_understanding)} rows")


### Additional imports for helper utilities

In [ ]:
import numpy as np
import re


def _norm_text(v: Any) -> str:
    s = "" if pd.isna(v) else str(v)
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def _split_atomic_rules(raw_text: Any) -> list[str]:
    """Split answer text into atomic rules with priority for numbered items."""
    s = "" if pd.isna(raw_text) else str(raw_text)
    s = s.replace("\r\n", "\n").replace("\r", "\n").strip()
    if not s:
        return []

    numbered = list(re.finditer(r"(?m)(^|\n)\s*(\d+)[\.)]\s*", s))
    parts = []
    if numbered:
        for i, m in enumerate(numbered):
            start = m.end()
            end = numbered[i + 1].start() if i + 1 < len(numbered) else len(s)
            chunk = s[start:end].strip(" \t\n-:;")
            if chunk:
                parts.append(chunk)
    else:
        lines = [ln.strip() for ln in s.split("\n") if ln.strip()]
        for ln in lines:
            ln = re.sub(r"^\s*(?:\d+[\.)]|[-*?])\s*", "", ln).strip(" \t-:;")
            if ln:
                parts.append(ln)
        if len(parts) == 1 and ";" in parts[0]:
            semi_parts = [p.strip() for p in parts[0].split(";") if p.strip()]
            if len(semi_parts) > 1:
                parts = semi_parts

    out = []
    seen = set()
    for item in parts:
        key = _norm_text(item)
        if key and key not in seen:
            out.append(item)
            seen.add(key)
    return out


def _safe_div(num: float, den: float) -> float:
    return float(num / den) if den else np.nan


def _serialize_ids(ids: list[int], prefix: str) -> str:
    if not ids:
        return ""
    return ",".join(f"{prefix}_{i}" for i in ids)


def _serialize_positions(ids: list[int]) -> str:
    if not ids:
        return ""
    return ",".join(str(i + 1) for i in ids)


def _serialize_texts(items: list[str]) -> str:
    if not items:
        return ""
    return " | ".join(items)


def _triage_zone(score: float, accept_threshold: float, reject_threshold: float) -> str:
    if pd.isna(score):
        return "auto_reject"
    if score >= accept_threshold:
        return "auto_accept"
    if score <= reject_threshold:
        return "auto_reject"
    return "uncertain"


def _needs_human_review(zone: str) -> int:
    return int(zone == "uncertain")

def _coverage_from_scores(
    col_scores: np.ndarray,
    match_threshold: float,
    strong_threshold: float,
) -> tuple[list[int], str, str]:
    if col_scores.size == 0:
        return [], "missing", ""

    matched_idx = [i for i, s in enumerate(col_scores) if float(s) >= float(match_threshold)]
    m = len(matched_idx)
    if m == 0:
        return matched_idx, "missing", ""
    if m == 1:
        q = "strong" if float(col_scores[matched_idx[0]]) >= float(strong_threshold) else "weak"
        return matched_idx, "exact", q

    all_strong = all(float(col_scores[i]) >= float(strong_threshold) for i in matched_idx)
    return matched_idx, "split", ("strong" if all_strong else "weak")


def _is_blank_series(sr: pd.Series) -> pd.Series:
    return sr.isna() | (sr.astype(str).str.strip() == "")


# Parse serialized predicted IDs such as "P_0,P_2" into integer indices [0, 2].
def _parse_pred_ids(value: Any) -> list[int]:
    s = "" if pd.isna(value) else str(value)
    if not s.strip():
        return []

    out = []
    for token in s.split(","):
        t = token.strip().upper()
        if t.startswith("P_"):
            t = t[2:]
        elif t.startswith("P"):
            t = t[1:]
        if t.isdigit():
            out.append(int(t))
    return out


def _greedy_one_to_one_assignment(scores: np.ndarray) -> dict[int, int]:
    """Greedy one-to-one assignment on similarity matrix (rows=pred, cols=gt)."""
    assignments: dict[int, int] = {}
    if scores.size == 0:
        return assignments

    n_pred, n_gt = scores.shape
    if n_pred == 0 or n_gt == 0:
        return assignments

    triples = []
    for pred_i in range(n_pred):
        for gt_i in range(n_gt):
            val = float(scores[pred_i, gt_i])
            if np.isnan(val):
                continue
            triples.append((val, pred_i, gt_i))
    triples.sort(key=lambda x: x[0], reverse=True)

    used_pred = set()
    used_gt = set()
    for _, pred_i, gt_i in triples:
        if pred_i in used_pred or gt_i in used_gt:
            continue
        assignments[pred_i] = gt_i
        used_pred.add(pred_i)
        used_gt.add(gt_i)
    return assignments


def _is_blank_value(v) -> bool:
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    return isinstance(v, str) and v.strip() == ""


def _sample_key(generated_question_id, row_index) -> str:
    gq = "" if _is_blank_value(generated_question_id) else str(generated_question_id)
    if _is_blank_value(row_index):
        ri = ""
    else:
        try:
            ri = str(int(float(row_index)))
        except Exception:
            ri = str(row_index)
    return f"{gq}||{ri}"


def _tokenize_text(s: str) -> list[str]:
    t = _norm_text(s)
    return t.split() if t else []



def _aggregate_rate(sample_df: pd.DataFrame, num_col: str, den_col: str) -> float:
    if sample_df.empty:
        return np.nan
    den = float(sample_df[den_col].sum())
    if den <= 0:
        return np.nan
    num = float(sample_df[num_col].sum())
    return float(num / den)


def _dataset_text_overall(n_questions: int) -> str:
    return f"N = {int(n_questions)} questions"


def _dataset_text_classification(n_questions: int, gt_rules: int | None) -> str:
    if gt_rules is None:
        return f"Q = {int(n_questions)}"
    return f"Q = {int(n_questions)}, GT rules = {int(gt_rules)}"


def _ngrams(tokens: list[str], n: int):
    if len(tokens) < n or n <= 0:
        return []
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]


### Metric helpers

In [ ]:
def _get_sbert_model():
    model_sbert = globals().get("_SCENE_SBERT_MODEL")
    if model_sbert is None:
        from sentence_transformers import SentenceTransformer

        model_sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        globals()["_SCENE_SBERT_MODEL"] = model_sbert
    return model_sbert


def _sbert_pairwise_scores(pred_texts: list[str], gt_texts: list[str]) -> np.ndarray:
    """Pairwise SBERT cosine similarity: rows=predictions, cols=ground-truth."""
    if len(pred_texts) == 0 or len(gt_texts) == 0:
        return np.empty((len(pred_texts), len(gt_texts)), dtype=float)

    try:
        from sentence_transformers import util

        model_sbert = _get_sbert_model()

        emb_pred = model_sbert.encode(
            pred_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        emb_gt = model_sbert.encode(
            gt_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        scores = util.cos_sim(emb_pred, emb_gt).cpu().numpy()
        return scores.astype(float)
    except Exception as e:
        print(f"Sentence-BERT pairwise similarity failed: {e}")
        return np.empty((len(pred_texts), len(gt_texts)), dtype=float)

def _safe_sbert_pairwise_scores(pred_texts: list[str], gt_texts: list[str]) -> np.ndarray:
    if len(pred_texts) == 0 or len(gt_texts) == 0:
        return np.empty((len(pred_texts), len(gt_texts)), dtype=float)
    try:
        from sentence_transformers import SentenceTransformer, util

        model = globals().get("_RULE_SBERT_MODEL")
        if model is None:
            model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
            globals()["_RULE_SBERT_MODEL"] = model
        emb_pred = model.encode(
            pred_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        emb_gt = model.encode(
            gt_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        return util.cos_sim(emb_pred, emb_gt).cpu().numpy().astype(float)
    except Exception as e:
        if "_RULE_SBERT_WARNED" not in globals():
            print(f"Sentence-BERT unavailable for rule-understanding metrics: {e}")
            globals()["_RULE_SBERT_WARNED"] = True
        return np.full((len(pred_texts), len(gt_texts)), np.nan, dtype=float)


def _mean_sbert_similarity_local(sub: pd.DataFrame) -> float:
    if sub.empty or "ground_truth_answer" not in sub.columns or "model_answer" not in sub.columns:
        return np.nan
    gt = sub["ground_truth_answer"].map(_norm_text)
    pred = sub["model_answer"].map(_norm_text)
    valid = (gt != "") & (pred != "")
    if valid.sum() == 0:
        return np.nan
    scores = _safe_sbert_pairwise_scores(pred[valid].tolist(), gt[valid].tolist())
    if scores.size == 0:
        return np.nan
    diag = np.diag(scores)
    if len(diag) == 0:
        return np.nan
    if np.isnan(diag).all():
        return np.nan
    return float(np.nanmean(diag))


def _lcs_len(a: list[str], b: list[str]) -> int:
    if not a or not b:
        return 0
    m = len(b)
    prev = [0] * (m + 1)
    for i in range(1, len(a) + 1):
        curr = [0] * (m + 1)
        ai = a[i - 1]
        for j in range(1, m + 1):
            if ai == b[j - 1]:
                curr[j] = prev[j - 1] + 1
            else:
                curr[j] = max(prev[j], curr[j - 1])
        prev = curr
    return prev[m]


def _rouge_l_f1(reference_text: str, predicted_text: str) -> float:
    ref = _tokenize_text(reference_text)
    pred = _tokenize_text(predicted_text)
    if not ref and not pred:
        return np.nan
    if not ref or not pred:
        return 0.0
    lcs = _lcs_len(ref, pred)
    if lcs == 0:
        return 0.0
    recall = lcs / len(ref)
    precision = lcs / len(pred)
    if recall + precision == 0:
        return 0.0
    return float(2 * recall * precision / (recall + precision))


def _bleu2(reference_text: str, predicted_text: str, eps: float = 1e-12) -> float:
    ref = _tokenize_text(reference_text)
    pred = _tokenize_text(predicted_text)
    if not ref and not pred:
        return np.nan
    if not pred:
        return 0.0
    if not ref:
        return 0.0

    precisions = []
    for n in [1, 2]:
        pred_counts = Counter(_ngrams(pred, n))
        ref_counts = Counter(_ngrams(ref, n))
        total = sum(pred_counts.values())
        if total == 0:
            precisions.append(eps)
            continue
        clipped = sum(min(cnt, ref_counts[ng]) for ng, cnt in pred_counts.items())
        p_n = clipped / total
        precisions.append(max(p_n, eps))

    c = len(pred)
    r = len(ref)
    bp = 1.0 if c > r else math.exp(1.0 - (r / max(c, 1)))
    score = bp * math.exp(sum(math.log(p) for p in precisions) / 2.0)
    return float(score)

### 1.4 Data preparation and processing functions

In [ ]:
def _prepare_rule_eval_df(input_path: str) -> tuple[pd.DataFrame, bool]:
    p = Path(input_path)
    if not p.exists():
        return pd.DataFrame(), True

    df_raw = pd.read_csv(p)
    required_cols = ["ground_truth_answer", "model_answer", "layer_id"]
    missing_cols = [c for c in required_cols if c not in df_raw.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns in {input_path}: {missing_cols}")
    df = df_raw.copy()
    df["ground_truth_answer"] = df["ground_truth_answer"].fillna("").astype(str)
    df["model_answer"] = df["model_answer"].fillna("").astype(str)
    df["layer_id"] = df["layer_id"].fillna("").astype(str).str.strip()
    df["_layer_id_norm"] = df["layer_id"].str.replace(".", "_", regex=False)

    if "generated_question_id" not in df.columns:
        df["generated_question_id"] = pd.NA
    if "row_index" not in df.columns:
        df["row_index"] = df.index
    if "question_text" not in df.columns:
        if "question" in df.columns:
            df["question_text"] = df["question"].fillna("").astype(str)
        else:
            df["question_text"] = ""
    df["sample_key"] = df.apply(
        lambda r: _sample_key(r.get("generated_question_id"), r.get("row_index")),
        axis=1,
    )
    return df, False


def _compute_auto_gt_coverage(df_cls: pd.DataFrame) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    rows = []
    pred_rule_map = {}
    sample_meta_rows = []

    for idx, row in df_cls.iterrows():
        generated_question_id = row.get("generated_question_id")
        row_index = row.get("row_index", idx)
        if _is_blank_value(row_index):
            row_index = idx
        sample_key = row.get("sample_key", _sample_key(generated_question_id, row_index))

        gt_text_full = row.get("ground_truth_answer", "")
        pred_text_full = row.get("model_answer", "")
        gt_rules = _split_atomic_rules(gt_text_full)
        pred_rules = _split_atomic_rules(pred_text_full)
        pred_rule_map[sample_key] = pred_rules

        sample_meta_rows.append(
            {
                "sample_key": sample_key,
                "generated_question_id": generated_question_id,
                "row_index": row_index,
                "question_text": row.get("question_text", ""),
                "full_ground_truth_answer": gt_text_full,
                "full_model_answer": pred_text_full,
            }
        )

        sim = _safe_sbert_pairwise_scores(pred_rules, gt_rules)
        for gt_i, gt_text in enumerate(gt_rules):
            col_scores = sim[:, gt_i] if (sim.size > 0 and len(pred_rules) > 0) else np.array([], dtype=float)
            best_similarity = float(np.max(col_scores)) if col_scores.size > 0 else np.nan
            matched_pred_ids, coverage_structure, coverage_quality = _coverage_from_scores(
                col_scores,
                RULE_MATCH_THRESHOLD,
                RULE_STRONG_MATCH_THRESHOLD,
            )
            matched_pred_texts = [pred_rules[p] for p in matched_pred_ids]
            rows.append(
                {
                    "sample_key": sample_key,
                    "generated_question_id": generated_question_id,
                    "row_index": row_index,
                    "layer_id": row.get("layer_id", RULE_CLASSIFICATION_LAYER),
                    "question_text": row.get("question_text", ""),
                    "full_ground_truth_answer": gt_text_full,
                    "full_model_answer": pred_text_full,
                    "gt_rule_id": int(gt_i),
                    "gt_rule_position": int(gt_i + 1),
                    "gt_text": gt_text,
                    "matched_pred_rule_ids": _serialize_ids(matched_pred_ids, "P"),
                    "matched_pred_positions": _serialize_positions(matched_pred_ids),
                    "matched_pred_texts": _serialize_texts(matched_pred_texts),
                    "best_similarity": best_similarity,
                    "matched_pred_count": int(len(matched_pred_ids)),
                    "coverage_structure": coverage_structure,
                    "coverage_quality": coverage_quality,
                }
            )

    gt_df = pd.DataFrame(rows)
    if gt_df.empty:
        gt_df = pd.DataFrame(
            columns=[
                "sample_key",
                "generated_question_id",
                "row_index",
                "layer_id",
                "question_text",
                "full_ground_truth_answer",
                "full_model_answer",
                "gt_rule_id",
                "gt_rule_position",
                "gt_text",
                "matched_pred_rule_ids",
                "matched_pred_positions",
                "matched_pred_texts",
                "best_similarity",
                "matched_pred_count",
                "coverage_structure",
                "coverage_quality",
            ]
        )

    sample_meta_df = pd.DataFrame(sample_meta_rows).drop_duplicates(subset=["sample_key"])
    return gt_df, pred_rule_map, sample_meta_df


def _merge_review_fields(auto_gt_df: pd.DataFrame, review_csv: str) -> pd.DataFrame:
    if auto_gt_df.empty:
        for c in ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]:
            if c not in auto_gt_df.columns:
                auto_gt_df[c] = ""
        return auto_gt_df

    for c in ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]:
        if c not in auto_gt_df.columns:
            auto_gt_df[c] = ""

    p = Path(review_csv)
    if not p.exists():
        return auto_gt_df

    prev = pd.read_csv(p)
    alias_map = {
        "human_coverage_structure": [
            "human_coverage_structure",
            "human_coverage_label",
            "Coverage Structure",
            "Human Coverage Label",
        ],
        "human_coverage_quality": [
            "human_coverage_quality",
            "Coverage Quality",
            "Human Coverage Quality",
        ],
        "human_best_pred_ids": [
            "human_best_pred_ids",
            "Human Best Pred IDs",
            "Human Best Read IDs",
            "Human Best Read Ids",
        ],
        "human_notes": ["human_notes", "Human Notes"],
    }
    for target, aliases in alias_map.items():
        if target not in prev.columns:
            prev[target] = ""
        if _is_blank_series(prev[target]).all():
            for alias in aliases:
                if alias in prev.columns:
                    prev[target] = prev[alias].fillna("")
                    if not _is_blank_series(prev[target]).all():
                        break

    for c in ["row_index", "gt_rule_id"]:
        if c in auto_gt_df.columns:
            auto_gt_df[c] = pd.to_numeric(auto_gt_df[c], errors="coerce").astype("Int64")
        if c in prev.columns:
            prev[c] = pd.to_numeric(prev[c], errors="coerce").astype("Int64")

    key_cols = ["row_index", "gt_rule_id"]
    if "generated_question_id" in auto_gt_df.columns and "generated_question_id" in prev.columns:
        if not _is_blank_series(auto_gt_df["generated_question_id"]).all():
            key_cols = ["generated_question_id"] + key_cols

    prev_keep = prev[key_cols + ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]].drop_duplicates(
        subset=key_cols
    )
    merged = auto_gt_df.merge(prev_keep, on=key_cols, how="left", suffixes=("", "__prev"))
    for c in ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]:
        cp = f"{c}__prev"
        if cp in merged.columns:
            mask = _is_blank_series(merged[c])
            merged.loc[mask, c] = merged.loc[mask, cp].fillna("")
            merged.drop(columns=[cp], inplace=True)
    return merged


def _finalize_gt_coverage(gt_df: pd.DataFrame) -> pd.DataFrame:
    out = gt_df.copy()
    if out.empty:
        for c in [
            "auto_coverage_structure",
            "auto_coverage_quality",
            "auto_best_pred_ids",
            "final_coverage_structure",
            "final_coverage_quality",
            "final_best_pred_ids",
        ]:
            if c not in out.columns:
                out[c] = ""
        return out

    out["auto_coverage_structure"] = out.get("coverage_structure", "").fillna("").astype(str).str.strip().str.lower()
    out["auto_coverage_quality"] = out.get("coverage_quality", "").fillna("").astype(str).str.strip().str.lower()
    out["auto_best_pred_ids"] = out.get("matched_pred_rule_ids", "").fillna("")

    human_struct = out.get("human_coverage_structure", "").fillna("").astype(str).str.strip().str.lower()
    human_quality = out.get("human_coverage_quality", "").fillna("").astype(str).str.strip().str.lower()
    human_best = out.get("human_best_pred_ids", "").fillna("").astype(str).str.strip()

    out["final_coverage_structure"] = np.where(human_struct != "", human_struct, out["auto_coverage_structure"])
    out["final_coverage_quality"] = np.where(human_quality != "", human_quality, out["auto_coverage_quality"])
    out["final_coverage_quality"] = np.where(out["final_coverage_structure"] == "missing", "", out["final_coverage_quality"])
    out["final_best_pred_ids"] = np.where(human_best != "", human_best, out["auto_best_pred_ids"])
    return out


def _compute_sample_metrics_from_final(gt_final: pd.DataFrame, pred_rule_map: dict, sample_meta_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if sample_meta_df.empty:
        return pd.DataFrame(
            columns=[
                "sample_key",
                "generated_question_id",
                "row_index",
                "gt_total",
                "pred_total",
                "unique_matched_pred_count",
                "gt_covered_count",
                "gt_missing_count",
                "gt_split_count",
                "gt_weak_count",
                "pred_extra_count",
                "coverage_recall",
                "split_rate",
                "missing_rule_rate",
                "extra_rule_rate",
            ]
        )

    gt_grouped = gt_final.groupby("sample_key", dropna=False) if not gt_final.empty else {}
    for _, meta in sample_meta_df.iterrows():
        sample_key = meta["sample_key"]
        gt_sub = gt_grouped.get_group(sample_key).copy() if (not gt_final.empty and sample_key in gt_grouped.groups) else pd.DataFrame()

        gt_total = int(len(gt_sub))
        # Final metrics are computed from adjudicated GT labels (final_*), not auto labels.
        if gt_total > 0:
            gt_struct = gt_sub["final_coverage_structure"].astype(str).str.strip().str.lower()
            gt_quality = gt_sub["final_coverage_quality"].astype(str).str.strip().str.lower()
            gt_covered_count = int(gt_struct.isin(["exact", "split"]).sum())
            gt_missing_count = int((gt_struct == "missing").sum())
            gt_split_count = int((gt_struct == "split").sum())
            gt_weak_count = int(((gt_struct != "missing") & (gt_quality == "weak")).sum())
            # Extra-rule counting is based on IDs used in final_best_pred_ids.
            # If one predicted fragment supports multiple GT rules, it is still counted once per sample.
            used_ids = {pid for v in gt_sub["final_best_pred_ids"].tolist() for pid in _parse_pred_ids(v)}
        else:
            gt_covered_count = 0
            gt_missing_count = 0
            gt_split_count = 0
            gt_weak_count = 0
            used_ids = set()

        pred_total = int(len(pred_rule_map.get(sample_key, [])))
        unique_matched_pred_count = int(len(used_ids))
        pred_extra_count = int(max(0, pred_total - unique_matched_pred_count))

        rows.append(
            {
                "sample_key": sample_key,
                "generated_question_id": meta.get("generated_question_id"),
                "row_index": meta.get("row_index"),
                "gt_total": gt_total,
                "pred_total": pred_total,
                "unique_matched_pred_count": unique_matched_pred_count,
                "gt_covered_count": gt_covered_count,
                "gt_missing_count": gt_missing_count,
                "gt_split_count": gt_split_count,
                "gt_weak_count": gt_weak_count,
                "pred_extra_count": pred_extra_count,
                "coverage_recall": _safe_div(gt_covered_count, gt_total),
                "split_rate": _safe_div(gt_split_count, gt_total),
                "missing_rule_rate": _safe_div(gt_missing_count, gt_total),
                "extra_rule_rate": _safe_div(pred_extra_count, pred_total),
            }
        )

    return pd.DataFrame(rows)


def _classification_rouge_per_rule(gt_final: pd.DataFrame, pred_rule_map: dict) -> float:
    if gt_final.empty:
        return np.nan

    scores = []
    for _, row in gt_final.iterrows():
        gt_text = row.get("gt_text", "")
        pred_ids = _parse_pred_ids(row.get("final_best_pred_ids", ""))
        pred_rules = pred_rule_map.get(row.get("sample_key"), [])
        selected = [pred_rules[i] for i in pred_ids if 0 <= i < len(pred_rules)]
        pred_text = " ".join(selected).strip()
        score = _rouge_l_f1(gt_text, pred_text)
        if not pd.isna(score):
            scores.append(float(score))
    return float(np.mean(scores)) if scores else np.nan


def _rule_picture_metrics(df_pic: pd.DataFrame) -> dict:
    if df_pic.empty:
        return {"ROUGE-L (avg per rule)": np.nan, "Sentence-BERT similarity": np.nan, "BLEU": np.nan, "n_questions": 0}

    rouge_scores = []
    bleu_scores = []
    for _, row in df_pic.iterrows():
        gt = row.get("ground_truth_answer", "")
        pred = row.get("model_answer", "")
        if _is_blank_value(gt) or _is_blank_value(pred):
            continue
        r = _rouge_l_f1(gt, pred)
        b = _bleu2(gt, pred)
        if not pd.isna(r):
            rouge_scores.append(float(r))
        if not pd.isna(b):
            bleu_scores.append(float(b))

    sbert = _mean_sbert_similarity_local(df_pic)
    return {
        "ROUGE-L (avg per rule)": float(np.mean(rouge_scores)) if rouge_scores else np.nan,
        "Sentence-BERT similarity": sbert,
        "BLEU": float(np.mean(bleu_scores)) if bleu_scores else np.nan,
        "n_questions": int(len(df_pic)),
    }


# 2. 2_1 intermediate tables

## 2.1 Set up export tables

In [ ]:
candidate_rows = []
pairwise_rows = []
gt_coverage_rows = []
sample_metrics_rows = []

candidate_columns = [
    "generated_question_id",
    "row_index",
    "layer_id",
    "question_text",
    "pred_rule_id",
    "pred_rule_position",
    "pred_text",
    "matched_gt_id",
    "matched_gt_position",
    "matched_gt_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "similarity_score",
    "triage_zone",
    "needs_human_review",
    "candidate_flag",
    "auto_match",
]
pairwise_columns = [
    "generated_question_id",
    "row_index",
    "layer_id",
    "question_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "gt_rule_id",
    "gt_rule_position",
    "gt_text",
    "pred_rule_id",
    "pred_rule_position",
    "pred_text",
    "similarity_score",
    "triage_zone",
    "needs_human_review",
    "candidate_flag",
    "auto_match",
]
gt_coverage_columns = [
    "generated_question_id",
    "row_index",
    "layer_id",
    "question_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "gt_rule_id",
    "gt_rule_position",
    "gt_text",
    "matched_pred_rule_ids",
    "matched_pred_positions",
    "matched_pred_texts",
    "best_similarity",
    "triage_zone",
    "needs_human_review",
    "matched_pred_count",
    "coverage_structure",
    "coverage_quality",
    "human_coverage_structure",
    "human_coverage_quality",
    "human_best_pred_ids",
    "human_notes",
    "auto_coverage_structure",
    "auto_coverage_quality",
    "auto_best_pred_ids",
    "final_coverage_structure",
    "final_coverage_quality",
    "final_best_pred_ids",
]
sample_metrics_columns = [
    "generated_question_id",
    "row_index",
    "layer_id",
    "question_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "gt_total",
    "pred_total",
    "unique_matched_pred_count",
    "gt_covered_count",
    "gt_missing_count",
    "gt_split_count",
    "gt_weak_count",
    "pred_extra_count",
    "coverage_recall",
    "split_rate",
    "missing_rule_rate",
    "extra_rule_rate",
    "pair_auto_accept_count",
    "pair_uncertain_count",
    "pair_auto_reject_count",
]


## 2.2 Prepare review tables (layer_id == "2_1")

In [ ]:
for idx, row in df_rule_classification.iterrows():
    generated_question_id = row.get("generated_question_id")
    row_index_value = row.get("row_index", idx)
    if pd.isna(row_index_value):
        row_index_value = idx
    layer_value = row.get(COL_LAYER, RULE_CLASSIFICATION_LAYER)
    question_text = "" if pd.isna(row.get("question_text", "")) else str(row.get("question_text", ""))
    full_ground_truth_answer = "" if pd.isna(row.get(COL_GT, "")) else str(row.get(COL_GT, ""))
    full_model_answer = "" if pd.isna(row.get(COL_PRED, "")) else str(row.get(COL_PRED, ""))

    gt_rules = _split_atomic_rules(full_ground_truth_answer)
    pred_rules = _split_atomic_rules(full_model_answer)
    sim = _sbert_pairwise_scores(pred_rules, gt_rules)

    # Raw audit matrix: pairwise similarity for each predicted/GT rule pair.
    for pred_i, pred_text in enumerate(pred_rules):
        for gt_i, gt_text in enumerate(gt_rules):
            score = float(sim[pred_i, gt_i]) if sim.size > 0 else np.nan
            pairwise_rows.append(
                {
                    "generated_question_id": generated_question_id,
                    "row_index": row_index_value,
                    "layer_id": layer_value,
                    "question_text": question_text,
                    "full_ground_truth_answer": full_ground_truth_answer,
                    "full_model_answer": full_model_answer,
                    "gt_rule_id": gt_i,
                    "gt_rule_position": gt_i + 1,
                    "gt_text": gt_text,
                    "pred_rule_id": pred_i,
                    "pred_rule_position": pred_i + 1,
                    "pred_text": pred_text,
                    "similarity_score": score,
                    "candidate_flag": int((not pd.isna(score)) and (score >= RULE_WEAK_MATCH_THRESHOLD)),
                    "auto_match": int((not pd.isna(score)) and (score >= RULE_MATCH_THRESHOLD)),
                }
            )

    # One-to-one assignment (pred -> gt) used only for candidate review table.
    pred_to_gt = _greedy_one_to_one_assignment(sim)
    if len(pred_rules) == 0:
        candidate_rows.append(
            {
                "generated_question_id": generated_question_id,
                "row_index": row_index_value,
                "layer_id": layer_value,
                "question_text": question_text,
                "pred_rule_id": np.nan,
                "pred_rule_position": np.nan,
                "pred_text": "",
                "matched_gt_id": np.nan,
                "matched_gt_position": np.nan,
                "matched_gt_text": "",
                "full_ground_truth_answer": full_ground_truth_answer,
                "full_model_answer": full_model_answer,
                "similarity_score": np.nan,
                "candidate_flag": 0,
                "auto_match": 0,
            }
        )
    else:
        for pred_i, pred_text in enumerate(pred_rules):
            if pred_i in pred_to_gt:
                gt_i = pred_to_gt[pred_i]
                best_score = float(sim[pred_i, gt_i]) if sim.size > 0 else np.nan
                matched_gt_id = int(gt_i)
                matched_gt_position = int(gt_i + 1)
                matched_gt_text = gt_rules[gt_i]
            else:
                best_score = np.nan
                matched_gt_id = np.nan
                matched_gt_position = np.nan
                matched_gt_text = ""

            candidate_rows.append(
                {
                    "generated_question_id": generated_question_id,
                    "row_index": row_index_value,
                    "layer_id": layer_value,
                    "question_text": question_text,
                    "pred_rule_id": pred_i,
                    "pred_rule_position": pred_i + 1,
                    "pred_text": pred_text,
                    "matched_gt_id": matched_gt_id,
                    "matched_gt_position": matched_gt_position,
                    "matched_gt_text": matched_gt_text,
                    "full_ground_truth_answer": full_ground_truth_answer,
                    "full_model_answer": full_model_answer,
                    "similarity_score": best_score,
                    "candidate_flag": int((not pd.isna(best_score)) and (best_score >= RULE_WEAK_MATCH_THRESHOLD)),
                    "auto_match": int((not pd.isna(best_score)) and (best_score >= RULE_MATCH_THRESHOLD)),
                }
            )


## 2.3 Materialize tables

In [ ]:
rule_classification_candidate_matches_df = pd.DataFrame(candidate_rows, columns=candidate_columns)
rule_classification_all_pairwise_df = pd.DataFrame(pairwise_rows, columns=pairwise_columns)
# GT-centric coverage/sample metrics are materialized in the adjudication step (single source of truth).
rule_classification_gt_coverage_summary_df = pd.DataFrame(gt_coverage_rows, columns=gt_coverage_columns)
rule_classification_sample_level_metrics_df = pd.DataFrame(sample_metrics_rows, columns=sample_metrics_columns)
rule_classification_pairwise_df = rule_classification_all_pairwise_df.copy()


## 2.4 Apply triage and reviewer placeholders

In [ ]:
# Three-zone triage for review artifacts.
if "similarity_score" in rule_classification_candidate_matches_df.columns:
    rule_classification_candidate_matches_df["triage_zone"] = rule_classification_candidate_matches_df["similarity_score"].apply(
        lambda x: _triage_zone(x, RULE_AUTO_ACCEPT_THRESHOLD, RULE_AUTO_REJECT_THRESHOLD)
    )
    rule_classification_candidate_matches_df["needs_human_review"] = (
        rule_classification_candidate_matches_df["triage_zone"].map(_needs_human_review)
    )

if "similarity_score" in rule_classification_all_pairwise_df.columns:
    rule_classification_all_pairwise_df["triage_zone"] = rule_classification_all_pairwise_df["similarity_score"].apply(
        lambda x: _triage_zone(x, RULE_AUTO_ACCEPT_THRESHOLD, RULE_AUTO_REJECT_THRESHOLD)
    )
    rule_classification_all_pairwise_df["needs_human_review"] = (
        rule_classification_all_pairwise_df["triage_zone"].map(_needs_human_review)
    )
    rule_classification_pairwise_df = rule_classification_all_pairwise_df.copy()


## 2.5 Adjudication and sample-level metrics

In [ ]:
# Adjudication for candidate table (review helper fields only).
if "needs_human_review" in rule_classification_candidate_matches_df.columns:
    rule_classification_candidate_matches_df["human_review_required"] = (
        rule_classification_candidate_matches_df["needs_human_review"].fillna(0).astype(int)
    )
else:
    rule_classification_candidate_matches_df["human_review_required"] = 0

rule_classification_candidate_matches_df["auto_label"] = rule_classification_candidate_matches_df.get("auto_match", 0)
for col in ["human_label", "human_gt_assignment", "human_pred_assignment"]:
    if col not in rule_classification_candidate_matches_df.columns:
        rule_classification_candidate_matches_df[col] = ""
    else:
        rule_classification_candidate_matches_df[col] = rule_classification_candidate_matches_df[col].fillna("")

_human_label_norm = rule_classification_candidate_matches_df["human_label"].astype(str).str.strip()
rule_classification_candidate_matches_df["final_label"] = np.where(
    _human_label_norm != "",
    _human_label_norm,
    rule_classification_candidate_matches_df["auto_label"].astype(str),
)

_human_gt_norm = rule_classification_candidate_matches_df["human_gt_assignment"].astype(str).str.strip()
rule_classification_candidate_matches_df["final_gt_assignment"] = np.where(
    _human_gt_norm != "",
    _human_gt_norm,
    rule_classification_candidate_matches_df.get("matched_gt_id"),
)

_human_pred_norm = rule_classification_candidate_matches_df["human_pred_assignment"].astype(str).str.strip()
rule_classification_candidate_matches_df["final_pred_assignment"] = np.where(
    _human_pred_norm != "",
    _human_pred_norm,
    rule_classification_candidate_matches_df.get("pred_rule_id"),
)

# GT-centric path (single source of truth): auto coverage -> merge review -> final labels -> sample metrics.
rule_classification_gt_auto_df, rule_classification_pred_rule_map, rule_classification_sample_meta_df = _compute_auto_gt_coverage(
    df_rule_classification
)
rule_classification_gt_merged_df = _merge_review_fields(rule_classification_gt_auto_df, RULE_GT_COVERAGE_CSV)
rule_classification_gt_coverage_summary_df = _finalize_gt_coverage(rule_classification_gt_merged_df)

if "best_similarity" in rule_classification_gt_coverage_summary_df.columns:
    rule_classification_gt_coverage_summary_df["triage_zone"] = rule_classification_gt_coverage_summary_df["best_similarity"].apply(
        lambda x: _triage_zone(x, RULE_AUTO_ACCEPT_THRESHOLD, RULE_AUTO_REJECT_THRESHOLD)
    )
    rule_classification_gt_coverage_summary_df["needs_human_review"] = (
        rule_classification_gt_coverage_summary_df["triage_zone"].map(_needs_human_review)
    )
    rule_classification_gt_coverage_summary_df["human_review_required"] = (
        rule_classification_gt_coverage_summary_df["needs_human_review"].fillna(0).astype(int)
    )
else:
    rule_classification_gt_coverage_summary_df["human_review_required"] = 0

rule_classification_sample_level_metrics_df = _compute_sample_metrics_from_final(
    rule_classification_gt_coverage_summary_df,
    rule_classification_pred_rule_map,
    rule_classification_sample_meta_df,
)

rule_classification_sample_level_metrics_df["layer_id"] = RULE_CLASSIFICATION_LAYER
rule_classification_sample_level_metrics_df = rule_classification_sample_level_metrics_df.merge(
    rule_classification_sample_meta_df[["sample_key", "question_text", "full_ground_truth_answer", "full_model_answer"]],
    on="sample_key",
    how="left",
)


## 2.6 Sort, export, and reporting

In [ ]:
for col in ["pair_auto_accept_count", "pair_uncertain_count", "pair_auto_reject_count"]:
    if col not in rule_classification_sample_level_metrics_df.columns:
        rule_classification_sample_level_metrics_df[col] = 0

if (
    not rule_classification_all_pairwise_df.empty
    and {"generated_question_id", "row_index", "triage_zone"}.issubset(rule_classification_all_pairwise_df.columns)
):
    zone_counts = (
        rule_classification_all_pairwise_df
        .groupby(["generated_question_id", "row_index", "triage_zone"], dropna=False)
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    zone_counts = zone_counts.rename(
        columns={
            "auto_accept": "pair_auto_accept_count",
            "uncertain": "pair_uncertain_count",
            "auto_reject": "pair_auto_reject_count",
        }
    )
    for col in ["pair_auto_accept_count", "pair_uncertain_count", "pair_auto_reject_count"]:
        if col not in zone_counts.columns:
            zone_counts[col] = 0
    keep_cols = [
        "generated_question_id",
        "row_index",
        "pair_auto_accept_count",
        "pair_uncertain_count",
        "pair_auto_reject_count",
    ]
    rule_classification_sample_level_metrics_df = (
        rule_classification_sample_level_metrics_df
        .drop(columns=["pair_auto_accept_count", "pair_uncertain_count", "pair_auto_reject_count"], errors="ignore")
        .merge(zone_counts[keep_cols], on=["generated_question_id", "row_index"], how="left")
    )

for col in ["pair_auto_accept_count", "pair_uncertain_count", "pair_auto_reject_count"]:
    rule_classification_sample_level_metrics_df[col] = (
        rule_classification_sample_level_metrics_df[col].fillna(0).astype(int)
    )

# Stable ordering for manual review.
sort_cols = [
    c
    for c in ["generated_question_id", "row_index", "pred_rule_id"]
    if c in rule_classification_candidate_matches_df.columns
]
if sort_cols and not rule_classification_candidate_matches_df.empty:
    rule_classification_candidate_matches_df = (
        rule_classification_candidate_matches_df.sort_values(sort_cols).reset_index(drop=True)
    )

pair_sort_cols = [
    c
    for c in ["generated_question_id", "row_index", "pred_rule_id", "gt_rule_id"]
    if c in rule_classification_all_pairwise_df.columns
]
if pair_sort_cols and not rule_classification_all_pairwise_df.empty:
    rule_classification_all_pairwise_df = (
        rule_classification_all_pairwise_df.sort_values(pair_sort_cols).reset_index(drop=True)
    )
    rule_classification_pairwise_df = rule_classification_all_pairwise_df.copy()

gt_sort_cols = [c for c in ["generated_question_id", "row_index", "gt_rule_id"] if c in rule_classification_gt_coverage_summary_df.columns]
if gt_sort_cols and not rule_classification_gt_coverage_summary_df.empty:
    rule_classification_gt_coverage_summary_df = (
        rule_classification_gt_coverage_summary_df.sort_values(gt_sort_cols).reset_index(drop=True)
    )

sample_sort_cols = [c for c in ["generated_question_id", "row_index"] if c in rule_classification_sample_level_metrics_df.columns]
if sample_sort_cols and not rule_classification_sample_level_metrics_df.empty:
    rule_classification_sample_level_metrics_df = (
        rule_classification_sample_level_metrics_df.sort_values(sample_sort_cols).reset_index(drop=True)
    )

# unique_matched_pred_count: unique predicted IDs from final_best_pred_ids across all GT rows in a sample.
# One predicted fragment may support multiple GT rows but is counted once as used for extra-rule counting.
rule_classification_candidate_matches_df.to_csv(
    RULE_CLASSIFICATION_CANDIDATE_CSV,
    index=False,
    encoding="utf-8-sig",
)
rule_classification_all_pairwise_df.to_csv(
    RULE_ALL_PAIRWISE_CSV,
    index=False,
    encoding="utf-8-sig",
)
rule_classification_gt_coverage_summary_df.to_csv(
    RULE_GT_COVERAGE_CSV,
    index=False,
    encoding="utf-8-sig",
)
rule_classification_sample_level_metrics_df.to_csv(
    RULE_SAMPLE_LEVEL_METRICS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print(f"Rule classification rows (layer {RULE_CLASSIFICATION_LAYER}): {len(df_rule_classification)}")
print(
    f"Thresholds: match={RULE_MATCH_THRESHOLD}, "
    f"auto_accept={RULE_AUTO_ACCEPT_THRESHOLD}, auto_reject={RULE_AUTO_REJECT_THRESHOLD}"
)
print(f"Saved: {RULE_CLASSIFICATION_CANDIDATE_CSV} ({len(rule_classification_candidate_matches_df)} rows)")
print(f"Saved: {RULE_ALL_PAIRWISE_CSV} ({len(rule_classification_all_pairwise_df)} rows)")
print(f"Saved: {RULE_GT_COVERAGE_CSV} ({len(rule_classification_gt_coverage_summary_df)} rows)")
print(f"Saved: {RULE_SAMPLE_LEVEL_METRICS_CSV} ({len(rule_classification_sample_level_metrics_df)} rows)")

if "triage_zone" in rule_classification_all_pairwise_df.columns and not rule_classification_all_pairwise_df.empty:
    print("Pairwise triage zones:")
    print(rule_classification_all_pairwise_df["triage_zone"].value_counts(dropna=False).to_string())

rule_classification_candidate_matches_df.head(20)


# 3. 2_2 metrics\n\nThis section prepares the routing view for layer `2_2` (rule understanding from image context).\nThe actual `2_2` text metrics are computed later during model-bundle assembly (`_build_model_rule_understanding_bundle`) and then included in the final summary tables.

In [ ]:
# Routing mask for layer 2_2 is prepared here.
# Text metrics for rule picture reading are computed in the model bundle stage.

df_rule_understanding = df_eval[rule_understanding_mask].copy()

print(f"Rule understanding rows (layer {RULE_UNDERSTANDING_LAYER}): {len(df_rule_understanding)}")
print("Rule picture reading metrics are computed in the Save dictionary stage.")

In [ ]:
print("Rule classification candidate table shape:", rule_classification_candidate_matches_df.shape)
display(rule_classification_candidate_matches_df.head(10))

print("All pairwise similarity table shape:", rule_classification_all_pairwise_df.shape)
display(rule_classification_all_pairwise_df.head(10))

print("GT coverage summary table shape:", rule_classification_gt_coverage_summary_df.shape)
display(rule_classification_gt_coverage_summary_df.head(10))

print("Sample-level metrics table shape:", rule_classification_sample_level_metrics_df.shape)
display(rule_classification_sample_level_metrics_df.head(10))


In [ ]:
print("Section sizes (layer_id routing):")
print(f"Rule classification ({RULE_CLASSIFICATION_LAYER}):", len(df_rule_classification))
print(f"Rule understanding ({RULE_UNDERSTANDING_LAYER}):", len(df_rule_understanding))

In [ ]:
preview_cols = [
    "generated_question_id",
    "row_index",
    "layer_id",
    "question_text",
    "ground_truth_answer",
    "model_answer",
]
show_cols = [c for c in preview_cols if c in df_eval.columns]

if show_cols:
    display(df_eval.loc[rule_classification_mask, show_cols].head(10))
else:
    print("No preview columns found in df_eval.")

# 4. Assemble summary

In [ ]:
def _empty_rule_understanding_section_metrics() -> dict:
    return {
        "overall Sentence-BERT similarity": np.nan,
        "Rule classification": {
            "ROUGE-L (avg per rule)": np.nan,
            "Coverage Recall": np.nan,
            "Split Rate": np.nan,
            "Missing Rule Rate": np.nan,
            "Extra Rule Rate": np.nan,
        },
        "Rule picture reading": {
            "ROUGE-L (avg per rule)": np.nan,
            "Sentence-BERT similarity": np.nan,
            "BLEU": np.nan,
        },
        "dataset characteristics": {
            "overall": {"n_questions": 0, "text": _dataset_text_overall(0)},
            "rule_classification": {"n_questions": 0, "gt_rules": 0, "text": _dataset_text_classification(0, 0)},
            "rule_picture_reading": {"n_questions": 0, "text": _dataset_text_overall(0)},
        },
    }


def _build_rule_understanding_section_metrics(
    df_ru: pd.DataFrame,
    df_cls: pd.DataFrame,
    df_pic: pd.DataFrame,
    gt_final: pd.DataFrame,
    sample_metrics: pd.DataFrame,
    pred_rule_map: dict,
) -> dict:
    overall_sbert = _mean_sbert_similarity_local(df_ru)
    cls_rouge = _classification_rouge_per_rule(gt_final, pred_rule_map)
    cls_coverage_recall = _aggregate_rate(sample_metrics, "gt_covered_count", "gt_total")
    cls_split_rate = _aggregate_rate(sample_metrics, "gt_split_count", "gt_total")
    cls_missing_rate = _aggregate_rate(sample_metrics, "gt_missing_count", "gt_total")
    cls_extra_rate = _aggregate_rate(sample_metrics, "pred_extra_count", "pred_total")

    pic_metrics = _rule_picture_metrics(df_pic)
    cls_gt_rules = int(sample_metrics["gt_total"].sum()) if not sample_metrics.empty else int(len(gt_final))
    ds_overall = {"n_questions": int(len(df_ru)), "text": _dataset_text_overall(int(len(df_ru)))}
    ds_cls = {
        "n_questions": int(len(df_cls)),
        "gt_rules": cls_gt_rules,
        "text": _dataset_text_classification(int(len(df_cls)), cls_gt_rules),
    }
    ds_pic = {"n_questions": int(len(df_pic)), "text": _dataset_text_overall(int(len(df_pic)))}

    return {
        "overall Sentence-BERT similarity": overall_sbert,
        "Rule classification": {
            "ROUGE-L (avg per rule)": cls_rouge,
            "Coverage Recall": cls_coverage_recall,
            "Split Rate": cls_split_rate,
            "Missing Rule Rate": cls_missing_rate,
            "Extra Rule Rate": cls_extra_rate,
        },
        "Rule picture reading": {
            "ROUGE-L (avg per rule)": pic_metrics["ROUGE-L (avg per rule)"],
            "Sentence-BERT similarity": pic_metrics["Sentence-BERT similarity"],
            "BLEU": pic_metrics["BLEU"],
        },
        "dataset characteristics": {
            "overall": ds_overall,
            "rule_classification": ds_cls,
            "rule_picture_reading": ds_pic,
        },
    }


def _build_model_rule_understanding_bundle(model_name: str, input_path: str, use_review_csv: bool = False) -> dict:
    df_model, missing = _prepare_rule_eval_df(input_path)
    if missing:
        return {
            "model": model_name,
            "missing_input": True,
            "input_path": input_path,
            "section_metrics": _empty_rule_understanding_section_metrics(),
            "artifacts": {},
        }

    ru_mask = df_model["_layer_id_norm"].isin([RULE_CLASSIFICATION_LAYER, RULE_UNDERSTANDING_LAYER])
    df_ru = df_model[ru_mask].copy()
    df_cls = df_model[df_model["_layer_id_norm"] == RULE_CLASSIFICATION_LAYER].copy()
    df_pic = df_model[df_model["_layer_id_norm"] == RULE_UNDERSTANDING_LAYER].copy()

    gt_auto, pred_rule_map, sample_meta_df = _compute_auto_gt_coverage(df_cls)
    if use_review_csv:
        gt_merged = _merge_review_fields(gt_auto, RULE_GT_COVERAGE_CSV)
    else:
        gt_merged = gt_auto.copy()
        for c in ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]:
            if c not in gt_merged.columns:
                gt_merged[c] = ""
    gt_final = _finalize_gt_coverage(gt_merged)
    sample_metrics = _compute_sample_metrics_from_final(gt_final, pred_rule_map, sample_meta_df)

    section_metrics = _build_rule_understanding_section_metrics(
        df_ru,
        df_cls,
        df_pic,
        gt_final,
        sample_metrics,
        pred_rule_map,
    )

    return {
        "model": model_name,
        "missing_input": False,
        "input_path": input_path,
        "section_metrics": section_metrics,
        "artifacts": {
            "df_rule_understanding_all": df_ru,
            "df_rule_classification": df_cls,
            "df_rule_picture_reading": df_pic,
            "gt_coverage_final": gt_final,
            "sample_level_metrics": sample_metrics,
        },
    }


## 4.1 Build cross-model bundles

### 4.2 Populate model metrics

In [ ]:
RULE_UNDERSTANDING_MODEL_BUNDLES = {}
RULE_UNDERSTANDING_MODEL_METRICS = {}

if "MODEL_METRICS" not in globals() or not isinstance(MODEL_METRICS, dict):
    MODEL_METRICS = {}

for _model_name in RULE_MODEL_COLUMNS:
    _input_path = RULE_MODEL_INPUTS.get(_model_name, "")
    _use_review = _model_name == "Claude Opus 4.6"

    # Reuse already computed Claude reviewed artifacts to avoid recomputing the same path.
    _has_precomputed_claude = all(
        k in globals()
        for k in [
            "df_eval",
            "rule_classification_gt_coverage_summary_df",
            "rule_classification_sample_level_metrics_df",
            "rule_classification_pred_rule_map",
        ]
    )
    if _model_name == "Claude Opus 4.6" and _has_precomputed_claude:
        _df_model = globals()["df_eval"].copy()
        _ru_mask = _df_model["_layer_id_norm"].isin([RULE_CLASSIFICATION_LAYER, RULE_UNDERSTANDING_LAYER])
        _df_ru = _df_model[_ru_mask].copy()
        _df_cls = _df_model[_df_model["_layer_id_norm"] == RULE_CLASSIFICATION_LAYER].copy()
        _df_pic = _df_model[_df_model["_layer_id_norm"] == RULE_UNDERSTANDING_LAYER].copy()
        _gt_final = globals()["rule_classification_gt_coverage_summary_df"].copy()
        _sample_metrics = globals()["rule_classification_sample_level_metrics_df"].copy()
        _pred_rule_map = globals()["rule_classification_pred_rule_map"]
        _bundle = {
            "model": _model_name,
            "missing_input": False,
            "input_path": _input_path,
            "section_metrics": _build_rule_understanding_section_metrics(
                _df_ru,
                _df_cls,
                _df_pic,
                _gt_final,
                _sample_metrics,
                _pred_rule_map,
            ),
            "artifacts": {
                "df_rule_understanding_all": _df_ru,
                "df_rule_classification": _df_cls,
                "df_rule_picture_reading": _df_pic,
                "gt_coverage_final": _gt_final,
                "sample_level_metrics": _sample_metrics,
            },
        }
    else:
        _bundle = _build_model_rule_understanding_bundle(
            _model_name,
            _input_path,
            use_review_csv=_use_review,
        )
    RULE_UNDERSTANDING_MODEL_BUNDLES[_model_name] = _bundle
    RULE_UNDERSTANDING_MODEL_METRICS[_model_name] = _bundle["section_metrics"]

    if _model_name not in MODEL_METRICS or not isinstance(MODEL_METRICS.get(_model_name), dict):
        MODEL_METRICS[_model_name] = {}
    MODEL_METRICS[_model_name][RULE_UNDERSTANDING_TITLE] = _bundle["section_metrics"]

CLAUDE_RULE_UNDERSTANDING_METRICS = RULE_UNDERSTANDING_MODEL_METRICS.get("Claude Opus 4.6")
GEMINI_RULE_UNDERSTANDING_METRICS = RULE_UNDERSTANDING_MODEL_METRICS.get("Gemini 3 Pro Preview")
GPT52_RULE_UNDERSTANDING_METRICS = RULE_UNDERSTANDING_MODEL_METRICS.get("GPT-5.2")

for _model_name in RULE_MODEL_COLUMNS:
    _b = RULE_UNDERSTANDING_MODEL_BUNDLES[_model_name]
    _ds = _b["section_metrics"]["dataset characteristics"]
    if _b.get("missing_input"):
        print(f"{_model_name}: missing {_b.get('input_path')}. Metrics left blank.")
    else:
        print(
            f"{_model_name}: "
            f"{_ds['overall']['text']} | "
            f"classification -> {_ds['rule_classification']['text']} | "
            f"picture reading -> {_ds['rule_picture_reading']['text']}"
        )

RULE_UNDERSTANDING_MODEL_METRICS


### 4.3 Assemble summary table

In [ ]:
import pandas as pd

SUMMARY_COLUMNS = [
    "Metric",
    "Claude Opus 4.6",
    "Gemini 3 Pro Preview",
    "GPT-5.2",
    "Dataset characteristics",
]


def _metric_value(model_name: str, block_name: str, metric_name: str):
    sec = RULE_UNDERSTANDING_MODEL_METRICS.get(model_name, {})
    if not isinstance(sec, dict):
        return ""
    if block_name == "overall":
        return sec.get("overall Sentence-BERT similarity", "")
    if block_name == "rule_classification":
        return sec.get("Rule classification", {}).get(metric_name, "")
    if block_name == "rule_picture_reading":
        return sec.get("Rule picture reading", {}).get(metric_name, "")
    return ""


def _dataset_text(block_name: str) -> str:
    # Prefer Claude as reference dataset column, then first non-empty available model.
    for model_name in RULE_MODEL_COLUMNS:
        sec = RULE_UNDERSTANDING_MODEL_METRICS.get(model_name, {})
        ds = sec.get("dataset characteristics", {}) if isinstance(sec, dict) else {}
        if block_name == "overall":
            txt = ds.get("overall", {}).get("text", "")
        elif block_name == "rule_classification":
            txt = ds.get("rule_classification", {}).get("text", "")
        elif block_name == "rule_picture_reading":
            txt = ds.get("rule_picture_reading", {}).get("text", "")
        else:
            txt = ""
        if not _is_blank_value(txt):
            return str(txt)
    return ""


row_specs = [
    {"type": "metric", "label": "overall Sentence-BERT similarity", "block": "overall", "metric": "overall Sentence-BERT similarity"},
    {"type": "label", "label": "Rule classification", "block": "rule_classification"},
    {"type": "metric", "label": "ROUGE-L (avg per rule)", "block": "rule_classification", "metric": "ROUGE-L (avg per rule)"},
    {"type": "metric", "label": "Coverage Recall", "block": "rule_classification", "metric": "Coverage Recall"},
    {"type": "metric", "label": "Split Rate", "block": "rule_classification", "metric": "Split Rate"},
    {"type": "metric", "label": "Missing Rule Rate", "block": "rule_classification", "metric": "Missing Rule Rate"},
    {"type": "metric", "label": "Extra Rule Rate", "block": "rule_classification", "metric": "Extra Rule Rate"},
    {"type": "label", "label": "Rule picture reading", "block": "rule_picture_reading"},
    {"type": "metric", "label": "ROUGE-L (avg per rule)", "block": "rule_picture_reading", "metric": "ROUGE-L (avg per rule)"},
    {"type": "metric", "label": "Sentence-BERT similarity", "block": "rule_picture_reading", "metric": "Sentence-BERT similarity"},
    {"type": "metric", "label": "BLEU", "block": "rule_picture_reading", "metric": "BLEU"},
]

summary_rows = []
for spec in row_specs:
    row = {"Metric": spec["label"]}
    if spec["type"] == "label":
        for model_name in RULE_MODEL_COLUMNS:
            row[model_name] = ""
        row["Dataset characteristics"] = ""
    else:
        for model_name in RULE_MODEL_COLUMNS:
            row[model_name] = _metric_value(model_name, spec["block"], spec["metric"])
        row["Dataset characteristics"] = _dataset_text(spec["block"])
    summary_rows.append(row)

rule_understanding_summary_df = pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS)
rule_understanding_summary_df.to_csv(RULE_UNDERSTANDING_SUMMARY_CSV, index=False, encoding="utf-8-sig")
print("Saved:", RULE_UNDERSTANDING_SUMMARY_CSV)
print("Title:", RULE_UNDERSTANDING_TITLE)
rule_understanding_summary_df


# 5. Export CSV/XLSX

### 5.1 Export configuration and paths

In [ ]:
import pandas as pd
from pathlib import Path

xlsx_path = Path(RULE_UNDERSTANDING_SUMMARY_XLSX)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    rule_understanding_summary_df.to_excel(
        writer,
        sheet_name="Rule understanding",
        index=False,
    )

print(f"Saved: {xlsx_path}")
display(rule_understanding_summary_df)


### 5.2 Exported artifacts dictionary check

In [ ]:
# Rule-understanding metrics dictionary for all models.
RULE_UNDERSTANDING_MODEL_METRICS


In [ ]:
# Quick check of exported artifacts.
print("CSV:", RULE_UNDERSTANDING_SUMMARY_CSV)
print("XLSX:", RULE_UNDERSTANDING_SUMMARY_XLSX)


# 6. Final preview/checks

In [ ]:
# Final table (Rule understanding / Rule classification)
rule_understanding_summary_df
